# Planning

The **Planning** pattern enables an agent to decompose a complex, high-level goal into a structured sequence of actionable steps — and then execute that plan, adapting as new information emerges.

Think of a planning agent as a specialist to whom you delegate a complex objective. You define the *what* (the goal and constraints) but not the *how*. The agent must:
1. Understand the initial state and the goal state
2. Discover the optimal sequence of actions to connect them
3. Adapt the plan when obstacles arise (venue unavailable, API returns unexpected data, etc.)

**When to use planning vs. a fixed workflow:**
- Use **planning** when the _how_ needs to be discovered (open-ended research, novel problems)
- Use **fixed workflows** when the _how_ is already known (repeatable pipelines, well-defined processes)

The trade-off is flexibility vs. predictability. Planning agents are more powerful but less deterministic.

**Use cases:** deep research reports, employee onboarding automation, competitive analysis, multi-phase content generation, autonomous project management.

## Implementation with Flyte v2

This notebook implements a two-phase planning workflow using **Flyte v2 primitives**:
1. **Plan Generation task** — the LLM decomposes the goal into typed `PlanStep` objects
2. **Plan Execution task** — each step is executed sequentially, with live progress reporting

The plan is a typed dataclass flowing between tasks, making the planning decisions fully visible in the Flyte UI and reproducible across retries.

#### CrewAI vs Flyte v2 — Key Differences

| Aspect | CrewAI | Flyte v2 |
|--------|--------|----------|
| **Plan representation** | Implicit (agent prompt + sequential tasks) | Typed `ResearchPlan` dataclass — serializable, inspectable |
| **Plan visibility** | Agent verbose logs | Structured output in Flyte UI |
| **Phase separation** | Single crew with planning agent + execution agent | Two Flyte tasks connected by a workflow — plan generation and execution are distinct, retriable steps |
| **Checkpointing** | None | `@flyte.trace` per LLM call; plan object persisted between phases |
| **Adaptation** | Agent re-prompts itself | `PlanStep.status` field — completed steps are skipped on retry |
| **Observability** | Console logs | Live HTML report in Flyte UI showing step-by-step progress |

### 1. Install dependencies

In [ ]:
!uv pip install 'flyte[tui]' anthropic

### Start the devbox

If you haven't already, install the flyte package with the command above, then launch the local cluster:

In [ ]:
!flyte start devbox

### 2. Export your API key

In [ ]:
!flyte create secret ANTHROPIC_API_KEY --value sk-...


### 3. Import dependencies and configure the Flyte TaskEnvironment

In [ ]:
from __future__ import annotations

import json
import os
from dataclasses import dataclass, field
from datetime import timedelta

from anthropic import AsyncAnthropic
import flyte
import flyte.report

flyte.init_from_config()

_image = (
    flyte.Image.from_debian_base(name="planning-agent", python_version=(3, 12))
    .with_pip_packages("anthropic>=0.25.0")
)

planning_env = flyte.TaskEnvironment(
    name="planning_env",
    image=_image,
    resources=flyte.Resources(cpu="1", memory="2Gi"),
    secrets=[
        flyte.Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY"),
    ],
)


### 4. Define the data models

The plan is represented as structured, typed data — not a string of text. This is the key architectural decision in the Flyte v2 approach:

- **`PlanStep`** — a single actionable unit with a `status` field that survives retries
- **`ResearchPlan`** — the full decomposition of the goal, persisted in Flyte's object storage between the planning and execution tasks
- **`PlanResult`** — the final typed output with all step results and a synthesized summary

Because the plan is a dataclass (not an in-memory object), the execution task can resume from any completed step on retry — no re-planning needed.

In [ ]:
@dataclass
class PlanStep:
    """
    A single step in the agent's plan.

    The `status` field enables idempotent execution: if the execution task
    is retried after a partial failure, completed steps are skipped.
    The `result` field accumulates the LLM output for that step.
    """
    step_number: int
    title: str
    description: str
    expected_output: str
    status: str = "pending"      # "pending" | "completed" | "skipped"
    result: str = ""


@dataclass
class ResearchPlan:
    """
    The agent's structured decomposition of a goal into executable steps.

    Stored in Flyte's object storage between the planning task and the
    execution task, giving full data lineage: you can inspect exactly what
    plan was generated for any execution in the Flyte UI.
    """
    goal: str
    rationale: str              # Why this decomposition was chosen
    steps: list[PlanStep]
    total_steps: int


@dataclass
class PlanResult:
    """Final typed output of the planning workflow."""
    goal: str
    plan: ResearchPlan
    final_report: str
    steps_completed: int
    steps_skipped: int

### 5. Define traced LLM helpers

Two separate LLM calls, each wrapped in `@flyte.trace`:
- `_generate_plan` — asks the model to decompose the goal into structured steps (JSON output)
- `_execute_step` — asks the model to complete a single plan step with full context of prior steps

Using JSON mode for plan generation ensures we get a machine-readable plan that can be stored and inspected as a typed dataclass.

In [ ]:
PLANNER_SYSTEM = """\
You are an expert research planner. Given a research goal, decompose it into
3-6 concrete, sequential steps that collectively achieve the goal.

Each step should be:
- Specific and actionable (not vague)
- Executable by an LLM with general knowledge
- Clearly scoped (not overlapping with other steps)

Return ONLY valid JSON matching this schema:
{
  "rationale": "<why you chose this decomposition>",
  "steps": [
    {
      "step_number": 1,
      "title": "<short title>",
      "description": "<what to do in this step>",
      "expected_output": "<what the output of this step should look like>"
    }
  ]
}"""

EXECUTOR_SYSTEM = """\
You are a research specialist executing a specific step in a multi-step research plan.
You have the context of all previously completed steps to inform your work.
Produce thorough, accurate, well-structured output for your assigned step.
Cite key facts and figures where relevant."""


@flyte.trace
async def _generate_plan(goal: str) -> dict:
    """
    Traced plan generation call.

    Returns a dict matching the planner JSON schema.
    Using explicit JSON output ensures the plan is machine-readable
    and can be stored as a typed dataclass in Flyte's object storage.
    """
    client = AsyncAnthropic()
    response = await client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=2048,
        system=PLANNER_SYSTEM,
        messages=[{"role": "user", "content": f"Research goal: {goal}"}],
    )
    raw = response.content[0].text.strip()
    # Strip markdown fences if the model wraps the JSON
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    return json.loads(raw)


@flyte.trace
async def _execute_step(
    step: PlanStep,
    goal: str,
    completed_steps: list[PlanStep],
) -> str:
    """
    Traced step execution call.

    The executor receives the full context of completed steps so each step
    can build on prior findings — this is the same information-passing mechanism
    as LangGraph's state, but explicit and serializable.
    """
    client = AsyncAnthropic()

    prior_context = ""
    if completed_steps:
        prior_context = "\n\n".join(
            f"Step {s.step_number}: {s.title}\n{s.result}"
            for s in completed_steps
        )
        prior_context = f"\n\nPreviously completed steps for context:\n{prior_context}"

    user_message = (
        f"Overall research goal: {goal}\n\n"
        f"Your current step (Step {step.step_number}): {step.title}\n"
        f"Description: {step.description}\n"
        f"Expected output: {step.expected_output}"
        + prior_context
    )

    response = await client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=3000,
        system=EXECUTOR_SYSTEM,
        messages=[{"role": "user", "content": user_message}],
    )
    return response.content[0].text


@flyte.trace
async def _synthesize_report(goal: str, steps: list[PlanStep]) -> str:
    """Traced synthesis call. Combines all step outputs into a final report."""
    client = AsyncAnthropic()

    steps_summary = "\n\n".join(
        f"## {s.step_number}. {s.title}\n{s.result}"
        for s in steps if s.status == "completed"
    )

    response = await client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=4000,
        system="You are a professional research writer. Synthesize the provided research findings into a coherent, well-structured final report. Use clear headings, integrate findings across sections, and end with a concise executive summary.",
        messages=[{
            "role": "user",
            "content": f"Research goal: {goal}\n\nFindings from each step:\n{steps_summary}\n\nPlease synthesize these into a final report.",
        }],
    )
    return response.content[0].text

### 6. Define Phase 1: Plan Generation

The planning task is short and fast — it only makes one LLM call to decompose the goal. Its output is a fully typed `ResearchPlan` that Flyte stores in object storage before passing it to the execution task.

This phase separation is important for production reliability: if the execution task fails (e.g., API rate limit on step 4 of 5), you can retry _only the execution task_ — the expensive plan generation step is not re-run.

In [ ]:
@planning_env.task(
    retries=3,
    timeout=timedelta(minutes=5),
    # Enable caching for plan generation: the same goal should produce
    # a consistent plan structure. Cached plans make execution retries cheaper.
    cache=flyte.Cache(behavior="auto"),
)
async def generate_plan(goal: str) -> ResearchPlan:
    """
    Phase 1: Decompose the research goal into a structured plan.

    The output ResearchPlan is stored in Flyte's object storage and passed
    to the execution task. This clean phase boundary means:
    - The plan is visible and auditable before execution begins
    - Execution retries don't re-run (and re-pay for) plan generation
    - You can insert human review between planning and execution
    """
    plan_data = await _generate_plan(goal=goal)

    steps = [
        PlanStep(
            step_number=s["step_number"],
            title=s["title"],
            description=s["description"],
            expected_output=s["expected_output"],
        )
        for s in plan_data["steps"]
    ]

    return ResearchPlan(
        goal=goal,
        rationale=plan_data["rationale"],
        steps=steps,
        total_steps=len(steps),
    )

### 7. Define Phase 2: Plan Execution

The execution task runs each step sequentially, passing completed step results as context to subsequent steps. The live HTML report in the Flyte UI shows exactly which step is running, what output was produced, and the overall completion status — comparable to Google DeepResearch's transparent progress view.

In [ ]:
def _html_escape(text: str) -> str:
    return text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


def _build_report(plan: ResearchPlan, final_report: str = "") -> str:
    step_sections = []
    for step in plan.steps:
        color = {"completed": "green", "pending": "gray", "skipped": "orange"}.get(step.status, "gray")
        result_html = (
            f"<pre style='background:#f4f4f4;padding:1em;border-radius:4px;font-size:.85em'>"
            f"{_html_escape(step.result)}</pre>"
            if step.result else "<p><em>Pending...</em></p>"
        )
        step_sections.append(
            f"<section>"
            f"<h2>Step {step.step_number}: {_html_escape(step.title)} "
            f"<span style='color:{color};font-size:.8em'>[{step.status.upper()}]</span></h2>"
            f"<p><em>{_html_escape(step.description)}</em></p>"
            f"{result_html}</section><hr/>"
        )

    final_section = ""
    if final_report:
        final_section = (
            f"<section><h2 style='color:#1565c0'>Final Report</h2>"
            f"<pre style='background:#e3f2fd;padding:1em;border-radius:4px'>"
            f"{_html_escape(final_report)}</pre></section>"
        )

    return (
        "<html><head><style>"
        "body{font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;"
        "padding:1.5em;max-width:900px;margin:auto}"
        "h1{border-bottom:2px solid #333;padding-bottom:.4em}"
        "pre{white-space:pre-wrap;word-break:break-word}"
        "hr{border:none;border-top:1px solid #ddd;margin:1.5em 0}"
        "</style></head><body>"
        f"<h1>Planning Agent — Execution Report</h1>"
        f"<p><strong>Goal:</strong> {_html_escape(plan.goal)}</p>"
        f"<p><strong>Plan rationale:</strong> {_html_escape(plan.rationale)}</p>"
        + "".join(step_sections)
        + final_section
        + "</body></html>"
    )


@planning_env.task(
    retries=3,
    timeout=timedelta(minutes=30),
    cache=flyte.Cache(behavior="disable"),
    report=True,
)
async def execute_plan(plan: ResearchPlan) -> PlanResult:
    """
    Phase 2: Execute each step of the plan sequentially.

    Adaptation pattern:
      - Each step receives the full output of all prior steps as context.
      - If a step's result indicates the direction should change (e.g., a
        key assumption was wrong), subsequent steps naturally incorporate that
        — because they read the full prior context, not just the original prompt.
      - Steps with status="completed" are skipped on retry, making the
        execution idempotent.

    Observability:
      The live report updates after each step, showing real-time progress
      in the Flyte UI — comparable to DeepResearch's transparent execution view.
    """
    completed_steps: list[PlanStep] = []
    steps_skipped = 0

    # Emit initial report with all steps in pending state
    await flyte.report.replace.aio(_build_report(plan))
    await flyte.report.flush.aio()

    for step in plan.steps:
        # Idempotency: skip already-completed steps on retry
        if step.status == "completed":
            completed_steps.append(step)
            steps_skipped += 1
            continue

        # Execute the step with full prior context
        step.result = await _execute_step(
            step=step,
            goal=plan.goal,
            completed_steps=completed_steps,
        )
        step.status = "completed"
        completed_steps.append(step)

        # Update live report after each step completes
        await flyte.report.replace.aio(_build_report(plan))
        await flyte.report.flush.aio()

    # Synthesize all step outputs into a final report
    final_report = await _synthesize_report(goal=plan.goal, steps=plan.steps)

    # Final report update
    await flyte.report.replace.aio(_build_report(plan, final_report))
    await flyte.report.flush.aio()

    return PlanResult(
        goal=plan.goal,
        plan=plan,
        final_report=final_report,
        steps_completed=len(completed_steps) - steps_skipped,
        steps_skipped=steps_skipped,
    )

### 8. Connect the phases with a Flyte workflow

The workflow connects the two tasks with an explicit data edge: `generate_plan` → `execute_plan`. This makes the dependency visible in Flyte's DAG view and ensures:
- The plan is fully persisted before execution begins
- Each phase can be retried independently
- The plan object is inspectable in the UI between phases (useful for human review)

In [ ]:
@flyte.workflow
def planning_workflow(goal: str) -> PlanResult:
    """
    Two-phase planning workflow.

    DAG structure:
      generate_plan(goal) ──► execute_plan(plan) ──► PlanResult

    The plan object flows between tasks through Flyte's typed data plane,
    making it serialized, versioned, and fully auditable.
    """
    plan = generate_plan(goal=goal)
    return execute_plan(plan=plan)

### 9. Run locally

In [ ]:
research_goal = (
    "Produce a comprehensive analysis of the current state of quantum computing, "
    "covering: key technical milestones achieved in 2023-2024, leading companies "
    "and their approaches, primary applications being targeted, and remaining "
    "engineering challenges before practical quantum advantage."
)

run = flyte.run(
    planning_workflow,
    goal=research_goal,
)
run.wait()
result = run.outputs()[0]

print(f"Steps completed: {result.steps_completed}")
print(f"Steps skipped (from cache): {result.steps_skipped}")
print("\nPlan rationale:", result.plan.rationale)
print("\n" + "=" * 60)
print("FINAL REPORT")
print("=" * 60)
print(result.final_report)


### Running remotely

On a Flyte cluster, the two-phase workflow appears as a DAG: `generate_plan → execute_plan`. The plan object is visible as a structured output between the tasks. The execution task shows a live report tab with step-by-step progress.

1. Create the secret on the cluster:

In [ ]:
!flyte create secret ANTHROPIC_API_KEY --value sk-...

2. Switch to remote execution:

In [ ]:
run = flyte.run(planning_workflow, goal=research_goal)
run.wait()
result = run.outputs()[0]
print(result.final_report)


## Scaling the pattern

### Human-in-the-loop between planning and execution

A key advantage of separating plan generation from execution in a Flyte workflow is that you can insert a human review gate between the two tasks. This is exactly how Google DeepResearch works: the agent generates a plan, shows it to the user for review and modification, then executes it.

In Flyte, implement this with a `@flyte.gate` between the tasks — the workflow pauses at the gate waiting for human approval before proceeding to execution.

In [ ]:
# Human-in-the-loop planning workflow
@flyte.workflow
def planning_with_review(goal: str) -> PlanResult:
    """
    Planning workflow with a human review gate between phases.

    Execution flow:
      generate_plan(goal)
        ──► [HUMAN REVIEW GATE] ← reviewer approves/modifies plan in Flyte UI
        ──► execute_plan(approved_plan)
        ──► PlanResult

    The gate pauses the workflow until a human approves the plan in the
    Flyte UI. The reviewer can see the full plan structure (steps, rationale)
    as a structured output before approving execution.
    """
    plan = generate_plan(goal=goal)
    # Gate: workflow pauses here; reviewer inspects `plan` in the Flyte UI
    # and clicks Approve to continue (or modifies plan parameters before approving)
    approved = flyte.gate("review-plan", plan)
    return execute_plan(plan=approved)

### Parallel step execution

For plans where steps are independent (no data dependency between them), execute them in parallel using `asyncio.gather`. This is appropriate when each step answers a different sub-question that doesn't depend on prior answers.

In [ ]:
import asyncio

async def execute_plan_parallel(plan: ResearchPlan) -> list[PlanStep]:
    """
    Execute all plan steps concurrently when they are independent.

    Use this when steps don't depend on each other's outputs.
    Example: a plan with steps 'analyze market size', 'analyze competition',
    'analyze regulatory environment' — all can run in parallel.

    Contrast with sequential execution where step N reads step N-1's results
    (appropriate for iterative research or when steps build on each other).
    """
    results = await asyncio.gather(*[
        _execute_step(step=step, goal=plan.goal, completed_steps=[])
        for step in plan.steps
    ])
    for step, result in zip(plan.steps, results):
        step.result = result
        step.status = "completed"
    return plan.steps

## Scaling the pattern

The devbox runs each task in a fresh container. For production workloads with many short LLM calls, `ReusePolicy` eliminates cold-start overhead by keeping a pool of warm containers ready.

> **Note:** `ReusePolicy` is a Union-specific feature that requires a [Union deployment](https://www.union.ai/docs/v2/union/). It is not supported on the local devbox.

In [ ]:
# Requires a Union deployment — not supported on the local devbox
from datetime import timedelta

production_planning = flyte.TaskEnvironment(
    name="planning_prod",
    image=_image,
    resources=flyte.Resources(cpu="2", memory="4Gi"),
    secrets=[flyte.Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY")],
    reusable=flyte.ReusePolicy(
        replicas=(2, 8),
        concurrency=4,
        scaledown_ttl=timedelta(minutes=5),
        idle_ttl=timedelta(minutes=15),
    ),
)